In [ ]:
# Cell 1: Imports & Setup
import logging
import time
import json
import yaml
from pathlib import Path

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling_core.types.doc import ImageRefMode
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

# Set logging level
logging.basicConfig(level=logging.INFO)
_log = logging.getLogger(__name__)

print("✓ Imports complete")

In [ ]:
# Cell 2: Configuration
input_folder = Path("all_data")
output_base = Path("docling_outputs")

# Output subdirectories
output_json = output_base / "json"
output_md = output_base / "md"
output_yaml = output_base / "yaml"

# Configure PDF pipeline options
pdf_pipeline_options = PdfPipelineOptions()
pdf_pipeline_options.generate_page_images = True

print(f"Input folder: {input_folder}")
print(f"Output base: {output_base}")
print(f"PDF options: generate_page_images={pdf_pipeline_options.generate_page_images}")

In [ ]:
# Cell 3: Scan Input Files
# Find all files in input folder
all_files = list(input_folder.glob("*"))

# Filter out system/lock files and directories
input_files = [
    f for f in all_files 
    if f.is_file() 
    and not f.name.startswith('.')
    and not f.name.startswith('.~lock')
    and not f.name == '.DS_Store'
]

print(f"Found {len(input_files)} files to process:\n")
for i, file in enumerate(input_files, 1):
    print(f"{i:2d}. {file.name} ({file.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Cell 4: Initialize DocumentConverter
doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pdf_pipeline_options,
            backend=DoclingParseV4DocumentBackend
        )
    }
)

print("✓ DocumentConverter initialized with DoclingParseV4 backend")

In [ ]:
# Cell 5: Batch Convert All Files
print(f"Starting batch conversion of {len(input_files)} files...\n")

start_time = time.time()

# Convert all files (Docling auto-detects file types)
conv_results = doc_converter.convert_all(
    input_files,
    raises_on_error=False  # Continue processing even if some files fail
)

# Convert generator to list for reuse
conv_results_list = list(conv_results)

end_time = time.time()
total_time = end_time - start_time

print(f"\n✓ Batch conversion complete in {total_time:.2f} seconds")

In [ ]:
# Cell 6: Export Documents to Organized Folders
# Create output directories
output_json.mkdir(parents=True, exist_ok=True)
output_md.mkdir(parents=True, exist_ok=True)
output_yaml.mkdir(parents=True, exist_ok=True)

success_count = 0
partial_success_count = 0
failure_count = 0

file_details = []

for conv_res in conv_results_list:
    doc_filename = conv_res.input.file.stem
    file_detail = {
        "filename": conv_res.input.file.name,
        "status": conv_res.status.name,
        "errors": []
    }
    
    if conv_res.status == ConversionStatus.SUCCESS:
        success_count += 1
        
        # Export to JSON
        with (output_json / f"{doc_filename}.json").open("w", encoding="utf-8") as fp:
            fp.write(json.dumps(conv_res.document.export_to_dict()))
        
        # Export to Markdown
        with (output_md / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
            fp.write(conv_res.document.export_to_markdown())
        
        # Export to YAML
        with (output_yaml / f"{doc_filename}.yaml").open("w", encoding="utf-8") as fp:
            fp.write(yaml.safe_dump(conv_res.document.export_to_dict()))
        
        print(f"✓ Exported: {doc_filename}")
        
    elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
        partial_success_count += 1
        
        # Still export partial results
        with (output_json / f"{doc_filename}.json").open("w", encoding="utf-8") as fp:
            fp.write(json.dumps(conv_res.document.export_to_dict()))
        
        with (output_md / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
            fp.write(conv_res.document.export_to_markdown())
        
        with (output_yaml / f"{doc_filename}.yaml").open("w", encoding="utf-8") as fp:
            fp.write(yaml.safe_dump(conv_res.document.export_to_dict()))
        
        # Log errors
        for item in conv_res.errors:
            error_msg = item.error_message
            file_detail["errors"].append(error_msg)
            _log.warning(f"Partial success for {doc_filename}: {error_msg}")
        
        print(f"⚠ Partially exported: {doc_filename} ({len(conv_res.errors)} errors)")
        
    else:  # FAILURE
        failure_count += 1
        
        # Log errors
        for item in conv_res.errors:
            error_msg = item.error_message
            file_detail["errors"].append(error_msg)
        
        print(f"✗ Failed: {doc_filename}")
    
    file_details.append(file_detail)

print(f"\nExport Summary:")
print(f"  Success: {success_count}")
print(f"  Partial: {partial_success_count}")
print(f"  Failed:  {failure_count}")

In [ ]:
# Cell 7: Generate Conversion Report
conversion_report = {
    "summary": {
        "total_files": len(input_files),
        "success_count": success_count,
        "partial_success_count": partial_success_count,
        "failure_count": failure_count,
        "processing_time_seconds": round(total_time, 2)
    },
    "file_details": file_details
}

# Save report
report_path = output_base / "conversion_report.json"
with report_path.open("w", encoding="utf-8") as fp:
    json.dump(conversion_report, fp, indent=2)

print(f"✓ Conversion report saved to: {report_path}")

In [ ]:
# Cell 8: Display Summary & Preview
print("="*60)
print("BATCH CONVERSION SUMMARY")
print("="*60)
print(f"Total files processed:    {len(input_files)}")
print(f"Successful conversions:   {success_count}")
print(f"Partial conversions:      {partial_success_count}")
print(f"Failed conversions:       {failure_count}")
print(f"Total processing time:    {total_time:.2f} seconds")
print(f"Average time per file:    {total_time/len(input_files):.2f} seconds")
print("="*60)

# Show failed files if any
if failure_count > 0:
    print("\nFailed Files:")
    for detail in file_details:
        if detail["status"] == "FAILURE":
            print(f"  - {detail['filename']}")
            for error in detail["errors"]:
                print(f"    Error: {error}")

# Preview first successful markdown file
if success_count > 0 or partial_success_count > 0:
    md_files = list(output_md.glob("*.md"))
    if md_files:
        first_md = md_files[0]
        print(f"\nPreview of {first_md.name} (first 500 chars):")
        print("-" * 60)
        with first_md.open("r", encoding="utf-8") as fp:
            preview = fp.read(500)
            print(preview)
            if len(preview) == 500:
                print("...")
        print("-" * 60)

print(f"\n✓ All outputs saved to: {output_base.absolute()}")